In [0]:
print("BMW Vehicle Intelligence Platform")
print("Databricks setup successful!")

In [0]:
spark.sql("USE CATALOG bmw_vehicle_intelligence")
spark.sql("USE SCHEMA bronze")

print("Catalog and Bronze schema selected successfully!")

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS bmw_vehicle_intelligence.bronze.raw_data;

In [0]:
df=spark.read.csv("/Volumes/bmw_vehicle_intelligence/bronze/raw_data/bmw_vehicles")
display(df)

In [0]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

In [0]:
df.header=True
display(df)

In [0]:
print(df.columns)
display(df)

In [0]:
df_ = df.toDF(
    "vehicle_id",
    "customer_id",
    "vin",
    "model",
    "model_year",
    "fuel_type",
    "purchase_date"
)

display(df_)

In [0]:
print(df_.columns)

In [0]:
from pyspark.sql.functions import col

df= (
    df
    .withColumn("vehicle_id", col("vehicle_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("model_year", col("model_year").cast("int"))
    .withColumn("purchase_date", col("purchase_date").cast("date"))
)

df_.printSchema()

In [0]:
print(df.columns)

In [0]:
df = df.toDF(
    "vehicle_id",
    "customer_id",
    "vin",
    "model",
    "model_year",
    "fuel_type",
    "purchase_date"
)

display(df)

In [0]:
df = spark.read.csv("/Volumes/bmw_vehicle_intelligence/bronze/raw_data/bmw_vehicles")
display(df)

In [0]:
df = df.toDF(
    "vehicle_id",
    "customer_id",
    "vin",
    "model",
    "model_year",
    "fuel_type",
    "purchase_date"
)

print(df.columns)


In [0]:
from pyspark.sql.functions import col

df = (
    df
    .withColumn("vehicle_id", col("vehicle_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("model_year", col("model_year").cast("int"))
    .withColumn("purchase_date", col("purchase_date").cast("date"))
)

df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, upper

df = (
    df
    .withColumn("vin", upper(trim(col("vin"))))
    .withColumn("model", trim(col("model")))
    .withColumn("fuel_type", upper(trim(col("fuel_type"))))
)

display(df)

In [0]:
from pyspark.sql.functions import count

df.groupBy("vehicle_id") \
  .agg(count("*").alias("record_count")) \
  .filter(col("record_count") > 1) \
  .show()

In [0]:
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn("processed_at", current_timestamp())

display(df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bmw_vehicle_intelligence.silver")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bmw_vehicle_intelligence.silver.bmw_vehicle")

In [0]:
spark.sql("SHOW SCHEMAS IN bmw_vehicle_intelligence")

In [0]:
spark.sql("SHOW SCHEMAS IN bmw_vehicle_intelligence").show()

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS bmw_vehicle_intelligence.silver
""")

In [0]:
spark.sql("SHOW SCHEMAS IN bmw_vehicle_intelligence").show()

In [0]:
df.write .format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bmw_vehicle_intelligence.silver.bmw_vehicle")

In [0]:
spark.sql("""
SELECT * 
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
""").show()

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.vehicle_summary AS
SELECT
    model,
    fuel_type,
    COUNT(*) AS total_vehicles,
    AVG(model_year) AS average_model_year
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY model, fuel_type
""")

In [0]:
spark.sql("""
SELECT *
FROM bmw_vehicle_intelligence.gold.vehicle_summary
""").show()

In [0]:
spark.sql("""
SELECT
    SUM(total_vehicles) AS total_vehicles,
    COUNT(*) AS total_combinations
FROM bmw_vehicle_intelligence.gold.vehicle_summary
""").show()

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.model_summary AS
SELECT
    model,
    COUNT(*) AS total_vehicles,
    MIN(model_year) AS oldest_model_year,
    MAX(model_year) AS newest_model_year
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY model
ORDER BY total_vehicles DESC
""")

In [0]:
spark.sql("""
SELECT *
FROM bmw_vehicle_intelligence.gold.model_summary
""").show()

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.fuel_summary AS
SELECT
    fuel_type,
    COUNT(*) AS total_vehicles,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY fuel_type
ORDER BY total_vehicles DESC
""")

In [0]:
spark.sql("""
SELECT *
FROM bmw_vehicle_intelligence.gold.fuel_summary
""").show()

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.purchase_year_summary AS
SELECT
    YEAR(purchase_date) AS purchase_year,
    COUNT(*) AS total_purchases
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY YEAR(purchase_date)
ORDER BY purchase_year
""")

In [0]:
spark.sql("""
SELECT *
FROM bmw_vehicle_intelligence.gold.purchase_year_summary
""").show()

In [0]:
spark.sql("""
OPTIMIZE bmw_vehicle_intelligence.gold.vehicle_summary
""")

In [0]:
spark.sql("""
OPTIMIZE bmw_vehicle_intelligence.gold.model_summary
""")

In [0]:
spark.sql("""
OPTIMIZE bmw_vehicle_intelligence.gold.model_summary
""")

spark.sql("""
OPTIMIZE bmw_vehicle_intelligence.gold.fuel_summary
""")

spark.sql("""
OPTIMIZE bmw_vehicle_intelligence.gold.purchase_year_summary
""")

In [0]:
%sql
SELECT
    model,
    total_vehicles
FROM bmw_vehicle_intelligence.gold.model_summary
ORDER BY total_vehicles DESC;

### this is **_customer_**

In [0]:
df=spark.read.csv("/Volumes/bmw_vehicle_intelligence/bronze/raw_data/customers",header=True,inferSchema=True)
display(df)


In [0]:
df.printSchema()

In [0]:
df_ = df.toDF(
    "customer_name",
    "email",
    "phone",
    "city",
    "customer_id"
)

display(df_)

In [0]:
from pyspark.sql.functions import col, sum, when

df_.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_.columns
]).show()

In [0]:
from pyspark.sql.functions import count,col
df_.groupby("customer_id").agg(count("*").alias("record_count")).filter(col("record_count")>1).show()

In [0]:
from pyspark.sql.functions import col,trim,upper,lower,current_timestamp
df=(df_
    .withColumn("customer_name",trim(col("customer_name")))
    .withColumn("email",lower(trim(col("email"))))
    .withColumn("phone",col("phone").cast("string"))
    .withColumn("city",upper(trim(col("city"))))
    .withColumn("updated_date",current_timestamp())
    )
display(df)


In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("bmw_vehicle_intelligence.silver.customers")

In [0]:
customers_silver = spark.table(
    "bmw_vehicle_intelligence.silver.customers"
)

print("Total customer records:", customers_silver.count())

display(customers_silver)

In [0]:
spark.sql("""CREATE OR REPLACE TABLE bmw_vehicle_intelligence.silver.customers AS
SELECT city,count(*)AS total_customers from bmw_vehicle_intelligence.silver.customers  group by city ORDER BY  total_customers desc""")

In [0]:
display(spark.table("bmw_vehicle_intelligence.gold.customer_summary"))


THIS TABLE IS DEALERSHIP

In [0]:
df= spark.read.csv(
    "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/dealerships",
    header=False,
    inferSchema=True
)

display(df)

In [0]:
df.printSchema()

In [0]:
df_= df.toDF(
    "dealership_id",
    "dealership_name",
    "city",
    "state",
    "country"
)

display(df)

In [0]:
df.columns

In [0]:
df = df.toDF(
    "customer_id",
    "customer_name",
    "email",
    "phone",
    "city"
)

print(df.columns)
display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, upper, lower

df = (
    df
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("phone", trim(col("phone")))
    .withColumn("city", upper(trim(col("city"))))
)

display(df)

In [0]:
from pyspark.sql.functions import count

df.groupBy("customer_id") \
    .agg(count("*").alias("record_count")) \
    .filter(col("record_count") > 1) \
    .show()

In [0]:
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn(
    "processed_at",
    current_timestamp()
)

display(df)

In [0]:
df.write.format("delta") .mode("overwrite") .option("overwriteSchema", "true") .saveAsTable("bmw_vehicle_intelligence.silver.bmw_customer")

In [0]:
spark.sql("""
SELECT *
FROM bmw_vehicle_intelligence.silver.bmw_customer
LIMIT 10
""").show(truncate=False)

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.silver.bmw_dealership AS
SELECT
    customer_id AS dealership_id,
    customer_name AS dealership_name,
    email AS city,
    phone AS state,
    city AS country,
    processed_at
FROM bmw_vehicle_intelligence.silver.bmw_customer
""")

In [0]:
spark.sql("""
SELECT *
FROM bmw_vehicle_intelligence.silver.bmw_dealership
LIMIT 10
""").show(truncate=False)

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.dealership_summary AS
SELECT
    city,
    state,
    country,
    COUNT(DISTINCT dealership_id) AS total_dealerships
FROM bmw_vehicle_intelligence.silver.bmw_dealership
GROUP BY city, state, country
ORDER BY total_dealerships DESC
""")

In [0]:
display(
    spark.table(
        "bmw_vehicle_intelligence.gold.dealership_summary"
    )
)

In [0]:
display(dbutils.fs.ls("/Volumes/bmw_vehicle_intelligence/bronze/raw_data"))

In [0]:
files = dbutils.fs.ls(
    "/Volumes/bmw_vehicle_intelligence/bronze/raw_data"
)

for file in files:
    print(file.name)

### parts_inventory

In [0]:
df = spark.read.csv(
    "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/parts_inventory",
    header=False,
    inferSchema=True
)

print(df.columns)
df.show(5, truncate=False)

In [0]:
df = df.toDF(
    "part_id",
    "part_name",
    "category",
    "quantity",
    "unit_price",
    "supplier_name",
    "updated_at"
)

print(df.columns)
df.printSchema()

In [0]:
from pyspark.sql.functions import col, to_timestamp

df = (
    df
    .withColumn("part_id", col("part_id").cast("int"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("updated_at", to_timestamp(col("updated_at")))
)

df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, upper

df = (
    df
    .withColumn("part_name", trim(col("part_name")))
    .withColumn("category", upper(trim(col("category"))))
    .withColumn("supplier_name", trim(col("supplier_name")))
)

display(df)

In [0]:
from pyspark.sql.functions import count

df.groupBy("part_id") \
    .agg(count("*").alias("record_count")) \
    .filter(col("record_count") > 1) \
    .show()

In [0]:
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn(
    "processed_at",
    current_timestamp()
)

display(df)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "bmw_vehicle_intelligence.silver.bmw_parts_inventory"
    )

In [0]:
spark.sql("""
SELECT *
FROM bmw_vehicle_intelligence.silver.bmw_parts_inventory
LIMIT 10
""").show(truncate=False)

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.parts_inventory_summary AS
SELECT
    category,
    COUNT(*) AS total_part_types,
    SUM(quantity) AS total_quantity,
    ROUND(SUM(quantity * unit_price), 2) AS total_inventory_value,
    ROUND(AVG(unit_price), 2) AS average_unit_price,
    COUNT(DISTINCT supplier_name) AS total_suppliers
FROM bmw_vehicle_intelligence.silver.bmw_parts_inventory
GROUP BY category
""")

In [0]:
display(
    spark.table(
        "bmw_vehicle_intelligence.gold.parts_inventory_summary"
    )
)

### service_history

In [0]:
df = spark.read.csv(
    "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/service_history",
    header=False,
    inferSchema=True
)

display(df)

In [0]:
print(df.columns)
df.show(5, truncate=False)

In [0]:
df = df.toDF(
    "service_id",
    "vehicle_id",
    "service_date",
    "service_type",
    "dealership_name",
    "service_cost",
    "technician_id",
    "service_status",
    "description"
)

print(df.columns)

In [0]:
from pyspark.sql.functions import col, to_date

df = (
    df
    .withColumn("service_id", col("service_id").cast("int"))
    .withColumn("vehicle_id", col("vehicle_id").cast("int"))
    .withColumn("service_date", to_date(col("service_date")))
    .withColumn("service_cost", col("service_cost").cast("double"))
    .withColumn("technician_id", col("technician_id").cast("int"))
)

df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, upper

df = (
    df
    .withColumn("service_type", trim(col("service_type")))
    .withColumn("dealership_name", trim(col("dealership_name")))
    .withColumn("service_status", upper(trim(col("service_status"))))
    .withColumn("description", trim(col("description")))
)

display(df)

In [0]:
from pyspark.sql.functions import count, col

df.groupBy("service_id") \
  .agg(count("*").alias("record_count")) \
  .filter(col("record_count") > 1) \
  .show()

In [0]:
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()


In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn(
    "processed_at",
    current_timestamp()
)

display(df)

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bmw_vehicle_intelligence.silver.bmw_service_history")

In [0]:
silver_df = spark.table(
    "bmw_vehicle_intelligence.silver.bmw_service_history"
)

print("Total records:", silver_df.count())

display(silver_df)

In [0]:
%sql
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.service_summary AS
SELECT
    service_type,
    COUNT(*) AS total_services,
    COUNT(DISTINCT vehicle_id) AS total_vehicles_serviced,
    ROUND(SUM(service_cost), 2) AS total_service_revenue,
    ROUND(AVG(service_cost), 2) AS average_service_cost
FROM bmw_vehicle_intelligence.silver.bmw_service_history
GROUP BY service_type;

In [0]:
%sql
SELECT *
FROM bmw_vehicle_intelligence.gold.service_summary;

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT vehicle_id) AS unique_vehicles
FROM bmw_vehicle_intelligence.silver.bmw_service_history;

In [0]:
%sql
SELECT
    vehicle_id,
    COUNT(*) AS total_services
FROM bmw_vehicle_intelligence.silver.bmw_service_history
GROUP BY vehicle_id;

### WORK_ORDERS

In [0]:
df = spark.read.csv(
    "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/work_orders",
    header=False,
    inferSchema=True
)

print(df.columns)
df.show(5, truncate=False)

In [0]:
df.printSchema()

In [0]:
df = df.toDF(
    "work_order_id",
    "vehicle_id",
    "customer_id",
    "dealership_id",
    "created_date",
    "completion_date",
    "work_order_type",
    "labor_cost",
    "parts_cost",
    "total_cost",
    "work_order_status",
    "description"
)

df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, upper

df = (
    df
    .withColumn("work_order_type", trim(col("work_order_type")))
    .withColumn("work_order_status", upper(trim(col("work_order_status"))))
    .withColumn("description", trim(col("description")))
)

display(df)

In [0]:
from pyspark.sql.functions import count, col

df.groupBy("work_order_id") \
    .agg(count("*").alias("record_count")) \
    .filter(col("record_count") > 1) \
    .show()

In [0]:
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

In [0]:
%sql
SELECT
    work_order_status,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN completion_date IS NULL THEN 1 ELSE 0 END) AS missing_completion_dates
FROM bmw_vehicle_intelligence.bronze.work_orders
GROUP BY work_order_status;

In [0]:
from pyspark.sql.functions import col, sum, when, count

df.groupBy("work_order_status").agg(
    count("*").alias("total_orders"),
    sum(
        when(col("completion_date").isNull(), 1).otherwise(0)
    ).alias("missing_completion_dates")
).show()

In [0]:
df.filter(
    (col("work_order_status") == "COMPLETED") &
    (col("completion_date").isNull())
).select(
    "work_order_id",
    "vehicle_id",
    "work_order_status",
    "created_date",
    "completion_date"
).show(10, truncate=False)

In [0]:
df.filter(
    col("completion_date").isNotNull() &
    (col("completion_date") < col("created_date"))
).select(
    "work_order_id",
    "created_date",
    "completion_date",
    "work_order_status"
).show(10, truncate=False)

In [0]:
invalid_dates = df.filter(
    col("completion_date").isNotNull() &
    (col("completion_date") < col("created_date"))
)

print("Invalid date records:", invalid_dates.count())

In [0]:
from pyspark.sql.functions import when, lit

df = df.withColumn(
    "date_quality",
    when(
        col("completion_date").isNull(),
        lit("MISSING_COMPLETION_DATE")
    ).when(
        col("completion_date") < col("created_date"),
        lit("INVALID_DATE_SEQUENCE")
    ).otherwise(
        lit("VALID")
    )
)

display(df)

In [0]:
df.groupBy("date_quality") \
    .count() \
    .orderBy("date_quality") \
    .show()

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bmw_vehicle_intelligence.silver.bmw_work_orders"
    )

In [0]:
silver_df = spark.table(
    "bmw_vehicle_intelligence.silver.bmw_work_orders"
)

print("Total records:", silver_df.count())

silver_df.printSchema()

display(silver_df)

In [0]:
%sql

CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.work_order_summary AS
SELECT
    work_order_type,
    work_order_status,
    COUNT(*) AS total_work_orders,
    COUNT(DISTINCT vehicle_id) AS total_vehicles,
    ROUND(SUM(labor_cost), 2) AS total_labor_cost,
    ROUND(SUM(parts_cost), 2) AS total_parts_cost,
    ROUND(SUM(total_cost), 2) AS total_work_order_cost,
    ROUND(AVG(total_cost), 2) AS average_work_order_cost
FROM bmw_vehicle_intelligence.silver.bmw_work_orders
GROUP BY
    work_order_type,
    work_order_status;

In [0]:
%sql

SELECT *
FROM bmw_vehicle_intelligence.gold.work_order_summary;

In [0]:
%sql
SHOW TABLES IN bmw_vehicle_intelligence.gold;

In [0]:
%sql
SELECT COUNT(*) AS total_vehicles
FROM bmw_vehicle_intelligence.gold.vehicle_summary;

In [0]:
%sql
SELECT
    SUM(total_vehicles) AS total_bmw_vehicles
FROM bmw_vehicle_intelligence.gold.vehicle_summary;

In [0]:
%sql
SELECT
    SUM(total_vehicles) AS total_vehicles,
    SUM(total_vehicles) AS vehicle_count_check
FROM bmw_vehicle_intelligence.gold.vehicle_summary;

In [0]:
%sql
SELECT
 *
  
FROM bmw_vehicle_intelligence.gold.dealership_summary;